In [1]:
pip install transformers datasets evaluate seqeval torch matplotlib

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16282 sha256=f4d29a0bcde5949a7d04effd0aaf595bb069b96dc543ea0c600a1f8984e1b0b6
  Stored in directory: c:\users\himan\appdata\local\pip\cache\wheels\14\cf\a7\8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval

   -------------------- ------------------- 1/2 [evaluate]
   ---------------------------------------- 2/2 [evaluate]

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install -U datasets transformers evaluate

Note: you may need to restart the kernel to use updated packages.


In [6]:
!pip install -U datasets==2.19.0
!pip install -U huggingface_hub

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/542.0 kB ? eta -:--:--
   ---------------------------------------- 542.0/542.0 kB 10.1 MB/s  0:00:00

  Attempting uninstall: fsspec

    Found existing installation: fsspec 2025.10.0

    Uninstalling fsspec-2025.10.0:

      Successfully uninstalled fsspec-2025.10.0

   -------- ------------------------------- 1/5 [fsspec]
  Attempting uninstall: dill
   -------- ------------------------------- 1/5 [fsspec]
    Found existing installation: dill 0.4.1
   -------- ------------------------------- 1/5 [fsspec]
    Uninstalling dill-0.4.1:
   -------- ------------------------------- 1/5 [fsspec]
      Successfully uninstalled dill-0.4.1
   -------- ------------------------------- 1/5 [fsspec]
   ---------------- ----------------------- 2/5 [dill]
  Attempting uninstall: multiprocess
   ---------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
s3fs 2025.10.0 requires fsspec==2025.10.0, but you have fsspec 2024.3.1 which is incompatible.


   ---------------------------------------- 0.0/637.4 kB ? eta -:--:--
   ---------------------------------------- 637.4/637.4 kB 12.0 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0


In [1]:
from datasets import load_dataset

dataset = load_dataset("tner/conll2003")

print(dataset)

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3453
    })
})


In [2]:
from datasets import load_dataset

dataset = load_dataset("tner/conll2003")

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [4]:
label_list = [
    "O", "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]

In [5]:
def tokenize_and_align_labels(example):
    tokenized = tokenizer(example["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    word_ids = tokenized.word_ids()
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["tags"][word_idx])
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized["labels"] = labels
    return tokenized

In [6]:
tokenized_datasets = dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [7]:
small_train = tokenized_datasets["train"].select(range(2000))
small_val = tokenized_datasets["validation"].select(range(500))

In [8]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [9]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list)
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1
)

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator
)

In [12]:
trainer.train()

C:\Users\himan\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.195148


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.19514797973632814, metrics={'train_runtime': 200.1651, 'train_samples_per_second': 9.992, 'train_steps_per_second': 2.498, 'total_flos': 18743946857496.0, 'train_loss': 0.19514797973632814, 'epoch': 1.0})

In [13]:
!pip install seqeval evaluate

In [14]:
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=2)

    true_labels = []
    true_preds = []

    for pred, lab in zip(predictions, labels):
        curr_preds = []
        curr_labels = []

        for p_, l_ in zip(pred, lab):
            if l_ != -100:
                curr_preds.append(label_list[p_])
                curr_labels.append(label_list[l_])

        true_preds.append(curr_preds)
        true_labels.append(curr_labels)

    results = metric.compute(predictions=true_preds, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

In [ ]:
trainer.compute_metrics = compute_metrics
results = trainer.evaluate()

print("📊 Results:")
print(results)

In [17]:
from transformers import pipeline

nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

sentence = "John works at Google in California"

result = nlp(sentence)

for r in result:
    print(r["word"], "→", r["entity"])

john → LABEL_3
works → LABEL_0
at → LABEL_0
google → LABEL_1
in → LABEL_0
california → LABEL_5
